# MyVoice AI — External GPU Inference (Tier 1) & Private Fine-Tuning (Tier 2)

This notebook runs heavy voice-cloning workloads on an optional external GPU (Colab/Kaggle T4 or self-hosted CUDA) so the Android APK stays lightweight and **personal voice recordings NEVER enter the Git repository**.

> **Mandatory Privacy Rules**
> - Store ALL recordings, datasets, generated audio, speaker embeddings, and checkpoints in **separate private storage OUTSIDE `/content/myvoice-ai`** (e.g. `/content/drive/MyDrive/MyVoiceAI_Private` or `/kaggle/working/myvoice_private`).
> - Zero-shot inference uses 3–15s reference audio ephemerally without training a model.
> - Cloud upload or extraction of training data requires explicit user consent every time.
> - Use Cell 6 below to delete all remote private training data on demand.

### Verified Model Licenses
- **`ai4bharat/IndicF5` (Stage 1 Bengali)**: **MIT License**; gated Hugging Face repo.
- **`ResembleAI/chatterbox` (Stage 2 English evaluation)**: **MIT License**; archived/retired upstream repo, English-only (no Bengali support).
- **`SWivid/F5-TTS` (Optional future engine)**: Code MIT; pre-trained weights are **`CC-BY-NC-4.0` (Non-Commercial)**.

In [ ]:
# 1. Verify CUDA GPU & FFmpeg availability
!nvidia-smi
!ffmpeg -version | head -n 2

In [ ]:
# 2. Clone repository & configure SEPARATE private storage OUTSIDE the Git repo
import os
import secrets
from pathlib import Path
from getpass import getpass

%cd /content
REPO_DIR = Path('/content/myvoice-ai')
if not REPO_DIR.exists():
    !git clone https://github.com/kutub4594/myvoice-ai.git /content/myvoice-ai

# Configure private storage strictly OUTSIDE the Git repo
if Path('/content/drive/MyDrive').exists():
    PRIVATE_ROOT = Path('/content/drive/MyDrive/MyVoiceAI_Private')
elif Path('/kaggle/working').exists():
    PRIVATE_ROOT = Path('/kaggle/working/myvoice_private')
else:
    PRIVATE_ROOT = Path('/content/myvoice_private_storage')

assert REPO_DIR.resolve() not in PRIVATE_ROOT.resolve().parents and PRIVATE_ROOT.resolve() != REPO_DIR.resolve()
for sub in ('owners', 'datasets', 'embeddings', 'checkpoints', 'generated_audio', 'speaker_profiles', 'training_jobs'):
    (PRIVATE_ROOT / sub).mkdir(parents=True, exist_ok=True)

%cd /content/myvoice-ai/backend
!pip install -q -r requirements.txt -r requirements-ml.txt

# Never hardcode tokens or store voice data in the Git clone
os.environ['HF_TOKEN'] = getpass('Hugging Face Read Token (for ai4bharat/IndicF5): ')
os.environ['MYVOICE_API_TOKEN'] = getpass('API Auth Token (leave blank to auto-generate): ') or secrets.token_urlsafe(32)
os.environ['ALLOW_LOCAL_HTTP_DEV'] = 'false'
os.environ['VOICE_ENGINE'] = 'indicf5'
os.environ['DEVICE'] = 'cuda'
os.environ['PRIVATE_TRAINING_STORAGE_DIR'] = str(PRIVATE_ROOT)
print('Repo:', REPO_DIR, '| Private Storage (outside repo):', PRIVATE_ROOT)
print('Copy this API token into the mobile app Backend Settings (Bearer Auth):', os.environ['MYVOICE_API_TOKEN'])

In [ ]:
# 3. [Tier 1] Benchmark Bengali zero-shot synthesis or start API server
# All generated audio outputs go into PRIVATE_ROOT / 'generated_audio', NEVER inside the repo.
# Example live benchmark:
# !python scripts/benchmark_bengali.py \
#     --ref-audio "$PRIVATE_ROOT/speaker_profiles/clean_ref.wav" \
#     --ref-text 'আমি প্রতিদিন সকালে নিয়মিত বাংলা খবর পড়ি।' \
#     --out-dir "$PRIVATE_ROOT/generated_audio/benchmark"

# Optional: Start Uvicorn (supports /api/v1/synthesize, /api/v1/training/upload, DELETE /api/v1/training/data)
# !nohup uvicorn app.main:app --host 0.0.0.0 --port 8000 > /content/uvicorn.log 2>&1 &
print('Tier 1 GPU cells ready.')

In [ ]:
# 4. [Tier 2] Mount Google Drive (Colab) & separate Speaker Profiles from Training Jobs
import json
import uuid
from datetime import datetime, timezone

if Path('/content').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PRIVATE_ROOT = Path('/content/drive/MyDrive/MyVoiceAI_Private')
        os.environ['PRIVATE_TRAINING_STORAGE_DIR'] = str(PRIVATE_ROOT)
    except Exception:
        pass

PROFILES_DIR = PRIVATE_ROOT / 'speaker_profiles'
JOBS_DIR = PRIVATE_ROOT / 'training_jobs'
PROFILES_DIR.mkdir(parents=True, exist_ok=True)
JOBS_DIR.mkdir(parents=True, exist_ok=True)

JOB_ID = os.environ.get('MYVOICE_JOB_ID') or f'job_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:6]}'
JOB_DIR = JOBS_DIR / JOB_ID
(JOB_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
JOB_STATE_FILE = JOB_DIR / 'job_state.json'

if not JOB_STATE_FILE.exists():
    JOB_STATE_FILE.write_text(json.dumps({
        'job_id': JOB_ID,
        'status': 'dataset_pending',
        'base_model': 'ai4bharat/IndicF5',
        'latest_checkpoint': None,
        'epoch': 0,
        'private_storage_outside_repo': True,
    }, indent=2))

print('Resumable Training Job Directory (outside repo):', JOB_DIR)

In [ ]:
# 5. [Tier 2] Require explicit consent & ZIP-bomb safe extraction of private dataset.zip
import csv
from app.services.private_storage import store_training_dataset_zip

DATASET_ZIP = PRIVATE_ROOT / 'datasets' / 'dataset.zip'

if DATASET_ZIP.exists():
    consent = input(f'Type YES to confirm explicit consent to extract personal dataset {DATASET_ZIP}: ').strip()
    if consent != 'YES':
        raise PermissionError('Dataset extraction aborted: explicit consent not provided.')

    manifest = store_training_dataset_zip(
        PRIVATE_ROOT,
        DATASET_ZIP.read_bytes(),
        owner_id='colab_owner',
        consent_confirmed=True,
        dataset_id='colab_manual',
    )
    EXTRACT_DIR = PRIVATE_ROOT / 'owners' / 'colab_owner' / 'datasets' / 'colab_manual'
    metadata_path = EXTRACT_DIR / 'metadata.csv'
    assert metadata_path.exists(), 'metadata.csv missing from dataset.zip'
    with metadata_path.open('r', encoding='utf-8') as f:
        rows = [row for row in csv.reader(f, delimiter='|') if len(row) >= 2]
    print(f'Safely extracted {len(rows)} utterances into isolated private storage {EXTRACT_DIR}')
else:
    print(f'No dataset.zip found in {DATASET_ZIP}. Upload via HTTPS + explicit consent from the app.')

In [ ]:
# 6. [Privacy Control] User-controlled deletion of ALL remote private datasets, embeddings, checkpoints & generated audio
import shutil

confirm_wipe = input(f'Type DELETE_REMOTE_DATA to permanently erase all personal voice data inside {PRIVATE_ROOT}: ').strip()
if confirm_wipe == 'DELETE_REMOTE_DATA':
    for sub in ('owners', 'datasets', 'embeddings', 'checkpoints', 'generated_audio', 'speaker_profiles', 'training_jobs'):
        target = PRIVATE_ROOT / sub
        if target.exists():
            shutil.rmtree(target)
            target.mkdir(parents=True, exist_ok=True)
    print('All remote personal voice data, embeddings, and checkpoints deleted from:', PRIVATE_ROOT)
else:
    print('Remote deletion cancelled.')